# PLTreeSHAP + FastTreeSHAP — Depth Sweep (Notebook 5)

Runs **PLTreeSHAP + FastTreeSHAP** across all datasets and task types in the
`woodelfhd_depth_sweep_experiment`.
This notebook is **independent** and can run in parallel with the other method notebooks.
Notebook 04 merges all result files and generates the HTML report.

> ⚠️ **Python 3.8 required** — `fasttreeshap` and `pltreeshap` only support Python 3.8.
> This notebook uses a `micromamba` environment to run under Python 3.8 inside Colab.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Copies and prepares the `run_python38_scripts.bash` helper from Drive
3. Clones `treebranchmarks` and `woodelf_explainer` repos
4. Restores any previous partial results from Drive
5. Installs all dependencies into a Python 3.8 micromamba environment
6. Runs `woodelfhd_depth_sweep_experiment --method pltreeshap_fasttreeshap`
7. Writes partial results to Drive as `pltreeshap_fasttreeshap.json`

### Task routing
| Task | Library used |
|------|--------------|
| Background SHAP | `pltreeshap.PLTreeExplainer` |
| Background SHAP IV | `pltreeshap.PLTreeExplainer` |
| Path-Dependent SHAP | `fasttreeshap.TreeExplainer` (v2) |
| Path-Dependent SHAP IV | `fasttreeshap.TreeExplainer` (v1) |

### Depth limits (same as OriginalWoodelf / AAAI)
The approach shares the same crash profile as OriginalWoodelf:
MEMORY_CRASH at high depths (D≥18 for SHAP, D≥15 for interactions on most datasets).

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

# Folder on Drive where results are persisted
DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'pltreeshap_fasttreeshap.json'

# Path to the run_python38_scripts.bash helper on your Drive
BASH_SCRIPT_DRIVE_PATH = '/content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/PLTreeSHAP/run_python38_scripts.bash'

print(f'Method cache will be saved to: {DRIVE_RESULT_PATH}')

In [ ]:
# ── Step 3: Prepare the Python 3.8 bash helper ───────────────────────────────
import shutil
shutil.copy(BASH_SCRIPT_DRIVE_PATH, '/content/run_python38_scripts.bash')

# Fix Windows CRLF line endings if present
!sed -i 's/\r$//' /content/run_python38_scripts.bash
!chmod +x /content/run_python38_scripts.bash
print('Bash helper ready.')

In [ ]:
# ── Step 4: Clone repositories ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'pltreeshap_fasttreeshap.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

In [ ]:
# ── Step 6: Run the experiment under Python 3.8 ───────────────────────────────
# fasttreeshap and pltreeshap require Python 3.8; micromamba provides the env.
#
# --method pltreeshap_fasttreeshap : only PLTreeSHAPFastTreeSHAPApproach is timed
# --result_location                : saves partial results to Drive after each mission
#
# pip groups (executed in order):
#   1. Core scientific stack pinned for Python 3.8 compatibility
#   2. Cython + LightGBM
#   3. woodelf_explainer (editable, from cloned repo)
#   4. treebranchmarks (editable, from cloned repo)
#   5. fasttreeshap
#   6. pltreeshap (from GitHub)

!/content/run_python38_scripts.bash \
    --cwd /content/treebranchmarks \
    --pip "numpy==1.24.0 scipy pyarrow fastparquet pandas xgboost scikit-learn tqdm" \
    --pip "Cython lightgbm" \
    --pip "-e /content/woodelf_explainer" \
    --pip "-e /content/treebranchmarks" \
    --pip "fasttreeshap" \
    --pip "-v --no-build-isolation git+https://github.com/schufa-innovationlab/pltreeshap@main" \
    --run "-m benchmarks.woodelfhd_depth_sweep_experiment --method pltreeshap_fasttreeshap --result_location {DRIVE_RESULT_PATH}"

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')